# 01 Data Sync and Check

**Project**: AIGC Research Comprehension System  
**Purpose**: Prepare remote Google Drive storage, acquire the 100-paper corpus, and parse PDFs into evidence sections.

**Operational Strategy**:
- **Google Drive** is the ONLY persistent data store.
- **Colab** is the temporary compute runtime.
- **Safe to Rerun**: Notebook detects existing data and only performs missing work unless `RESET_DATA` is true.

In [1]:
# @title Configuration Flags
RESET_DATA = False # @param {type:"boolean"}
RUN_DOWNLOAD = True # @param {type:"boolean"}
RUN_FALLBACK_DOWNLOAD = True # @param {type:"boolean"}
RUN_PARSE = True # @param {type:"boolean"}

MAX_PDF_TARGET = 100
MIN_PDF_REQUIRED_FOR_PARSE = 70
STOP_AFTER_PDF_COUNT = 80

# Optional: Set your email for Unpaywall API
UNPAYWALL_EMAIL = "" # @param {type:"string"}

## 1. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

drive.mount("/content/drive", force_remount=True)
assert os.path.exists("/content/drive"), "Drive mount failed!"

Mounted at /content/drive


## 2. Define Paths

In [4]:
BASE = Path("/content/drive/MyDrive/AIGC")
NOTEBOOK_DIR = BASE / "Notebook"
DATA_DIR = BASE / "Data"

PDF_DIR = DATA_DIR / "pdfs"
TEI_DIR = DATA_DIR / "tei_xml"
DOWNLOAD_LOG_DIR = DATA_DIR / "download_logs"
PARSED_DIR = DATA_DIR / "parsed"
SECTIONS_DIR = DATA_DIR / "sections"
TABLES_DIR = DATA_DIR / "tables"
PARSE_LOG_DIR = DATA_DIR / "parse_logs"
PROBES_DIR = DATA_DIR / "probes"
REGISTRY_DIR = DATA_DIR / "registry"
REPORTS_DIR = DATA_DIR / "reports"
CHECKPOINT_DIR = DATA_DIR / "checkpoints"

REPO_URL = "https://github.com/IanJ332/AIGC_Fake_Detection"
REPO_DIR = Path("/content/AIGC_Fake_Detection")

## 3. Folder Management

In [5]:
import shutil

sub_dirs = [
    PDF_DIR, TEI_DIR, DOWNLOAD_LOG_DIR, PARSED_DIR, SECTIONS_DIR,
    TABLES_DIR, PARSE_LOG_DIR, PROBES_DIR, REGISTRY_DIR, REPORTS_DIR,
    CHECKPOINT_DIR
]

if RESET_DATA:
    print("RESET_DATA is True. Deleting and recreating subfolders...")
    for d in sub_dirs:
        if d.exists():
            shutil.rmtree(d)
        d.mkdir(parents=True, exist_ok=True)
else:
    for d in sub_dirs:
        d.mkdir(parents=True, exist_ok=True)
    NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print("Folders verified.")

Folders verified.


## 4. Resource Report

In [6]:
import sys
import psutil
import subprocess

print(f"Python: {sys.version}")
print(f"RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
try:
    gpu = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).decode().strip()
    print(f"GPU: {gpu}")
    print("WARNING: GPU is present but not needed for this notebook.")
except:
    print("GPU: None (Correct for this notebook)")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
RAM: 12.67 GB
GPU: None (Correct for this notebook)


## 5. Clone Repository

In [7]:
!rm -rf /content/AIGC_Fake_Detection
!git clone {REPO_URL} {REPO_DIR}
!pip install -q pandas tqdm pyyaml pymupdf requests

Cloning into '/content/AIGC_Fake_Detection'...
remote: Enumerating objects: 114, done.
remote: Counting objects: 100% (114/114), done.
remote: Compressing objects: 100% (88/88), done.
remote: Total 114 (delta 38), reused 94 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (114/114), 918.55 KiB | 5.02 MiB/s, done.
Resolving deltas: 100% (38/38), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 61.8 MB/s eta 0:00:00


## 6. Sync Metadata & Setup Symlinks

In [8]:
def sync_to_drive():
    # Copy lightweight files from repo to Drive
    mapping = {
        REPO_DIR / "corpus/manifest_100.csv": REGISTRY_DIR / "manifest_100.csv",
        REPO_DIR / "corpus/document_registry.csv": REGISTRY_DIR / "document_registry.csv",
        REPO_DIR / "corpus/parse_registry.csv": REGISTRY_DIR / "parse_registry.csv",
        REPO_DIR / "docs/day1_status.md": REPORTS_DIR / "day1_status.md",
        REPO_DIR / "docs/day2_status.md": REPORTS_DIR / "day2_status.md",
        REPO_DIR / "docs/day3_parse_report.md": REPORTS_DIR / "day3_parse_report.md"
    }
    for src, dst in mapping.items():
        if src.exists():
            shutil.copy(src, dst)

def setup_symlinks():
    data_subsets = [
        ("pdfs", PDF_DIR), ("tei_xml", TEI_DIR), ("download_logs", DOWNLOAD_LOG_DIR),
        ("parsed", PARSED_DIR), ("sections", SECTIONS_DIR), ("tables", TABLES_DIR),
        ("parse_logs", PARSE_LOG_DIR)
    ]
    for name, drive_path in data_subsets:
        local_path = REPO_DIR / "corpus" / name
        if local_path.exists() and not local_path.is_symlink():
            shutil.rmtree(local_path)
        if not local_path.exists():
            os.symlink(drive_path, local_path)

sync_to_drive()
setup_symlinks()
print("Metadata synced and symlinks established.")

Metadata synced and symlinks established.


## 7. Primary Acquisition (Direct Download)

In [9]:
if RUN_DOWNLOAD:
    %cd {REPO_DIR}
    !python -m src.acquire.download_documents --manifest corpus/manifest_100.csv
    !python -m src.acquire.validate_documents --registry corpus/document_registry.csv

    # Sync results back to Drive
    shutil.copy(REPO_DIR / "corpus/document_registry.csv", REGISTRY_DIR / "document_registry.csv")
    shutil.copy(REPO_DIR / "docs/day2_status.md", REPORTS_DIR / "day2_status.md")
    shutil.copy(REPO_DIR / "docs/day2_document_acquisition_report.md", REPORTS_DIR / "day2_document_acquisition_report.md")
    print("Direct download phase complete.")

/content/AIGC_Fake_Detection
Starting acquisition for 100 papers...
[1/100] P001: Detecting GAN generated Fake Images using Co-occurrence Matr...
  Trying manifest PDF: https://library.imaging.org/ei/articles/31/5/art00008
[2/100] P002: Synthbuster: Towards Detection of Diffusion Model Generated ...
  Trying manifest PDF: https://ieeexplore.ieee.org/ielx7/8782710/9006934/10334046.pdf
[3/100] P003: GenImage: A Million-Scale Benchmark for Detecting AI-Generat...
  Trying manifest PDF: https://arxiv.org/pdf/2306.08571
[4/100] P004: DIRE for Diffusion-Generated Image Detection...
  Trying manifest PDF: https://arxiv.org/pdf/2303.09295
[5/100] P005: AIGI-Holmes: Towards Explainable and Generalizable AI-Genera...
  Trying manifest PDF: https://arxiv.org/pdf/2507.02664
[6/100] P006: $\bf{D^3}$QE: Learning Discrete Distribution Discrepancy-awa...
  Trying manifest PDF: https://arxiv.org/pdf/2510.05891
[7/100] P007: CNN-generated images are surprisingly easy to spot... for no...
  Trying manife

## 8. Fallback Downloader

In [10]:
import pandas as pd
import requests
import hashlib
from tqdm import tqdm
import time
import json

USER_AGENT = "AIGC-Research-Colab/1.0 (mailto:ian@example.com)"
SKIP_DOMAINS = [
    "onlinelibrary.wiley.com", "dl.acm.org", "ieee.org", "link.springer.com",
    "mdpi.com", "ojs.aaai.org", "academic.oup.com", "sagepub.com"
]

def get_sha256(file_path):
    h = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            h.update(chunk)
    return h.hexdigest()

def download_fallback(url, path):
    try:
        res = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=20, stream=True)
        if res.status_code == 200:
            with open(path, "wb") as f:
                for chunk in res.iter_content(8192): f.write(chunk)
            # Basic PDF validation
            if os.path.getsize(path) < 1000: return False
            with open(path, "rb") as f: head = f.read(4)
            return b"%PDF" in head or "pdf" in res.headers.get("Content-Type", "").lower()
    except: pass
    return False

if RUN_FALLBACK_DOWNLOAD:
    # Safely get UNPAYWALL_EMAIL from globals
    email = globals().get("UNPAYWALL_EMAIL", "")

    reg_path = REGISTRY_DIR / "document_registry.csv"
    if reg_path.exists():
        df = pd.read_csv(reg_path)
        current_count = sum(df["pdf_downloaded"] == True)

        if current_count < STOP_AFTER_PDF_COUNT:
            print(f"Current PDF count: {current_count}. Starting fallback for failed papers...")

            for idx, row in tqdm(df.iterrows(), total=len(df)):
                if row["pdf_downloaded"] == True: continue
                if current_count >= STOP_AFTER_PDF_COUNT: break

                paper_id = row["paper_id"]
                target_path = PDF_DIR / f"{paper_id}.pdf"
                title = row["title"]
                doi = row.get("doi")

                success = False

                # A. arXiv normalization
                url = str(row.get("source_url", "")) + " " + str(row.get("pdf_url", ""))
                if "arxiv.org/abs/" in url:
                    arxiv_id = url.split("arxiv.org/abs/")[-1].split()[0]
                    success = download_fallback(f"https://arxiv.org/pdf/{arxiv_id}", target_path)

                # B. Semantic Scholar
                if not success:
                    try:
                        s2_res = requests.get(f"https://api.semanticscholar.org/graph/v1/paper/search?query={title}&fields=openAccessPdf,url&limit=1").json()
                        if "data" in s2_res and s2_res["data"]:
                            oa_url = s2_res["data"][0].get("openAccessPdf", {}).get("url")
                            if oa_url and not any(d in oa_url for d in SKIP_DOMAINS):
                                success = download_fallback(oa_url, target_path)
                    except: pass

                # C. Unpaywall
                if not success and doi and email:
                    try:
                        upw_res = requests.get(f"https://api.unpaywall.org/v2/{doi}?email={email}").json()
                        upw_url = upw_res.get("best_oa_location", {}).get("url_for_pdf")
                        if upw_url and not any(d in upw_url for d in SKIP_DOMAINS):
                            success = download_fallback(upw_url, target_path)
                    except: pass

                if success:
                    df.at[idx, "pdf_downloaded"] = True
                    df.at[idx, "source_used"] = "colab_fallback"
                    df.at[idx, "file_size_bytes"] = os.path.getsize(target_path)
                    df.at[idx, "sha256"] = get_sha256(target_path)
                    df.at[idx, "needs_manual_review"] = False
                    current_count += 1
                    time.sleep(1)

            df.to_csv(reg_path, index=False)
            print(f"Fallback complete. New PDF count: {current_count}")

            with open(CHECKPOINT_DIR / "fallback_download_done.json", "w") as f:
                json.dump({"timestamp": time.time(), "final_count": current_count}, f)
        else:
            print(f"PDF count {current_count} already meets STOP_AFTER_PDF_COUNT ({STOP_AFTER_PDF_COUNT}).")

Current PDF count: 72. Starting fallback for failed papers...


100%|██████████| 100/100 [00:05<00:00, 18.80it/s]

Fallback complete. New PDF count: 72


## 9. PDF Count Gate

In [11]:
reg_path = REGISTRY_DIR / "document_registry.csv"
pdf_files = list(PDF_DIR.glob("*.pdf"))
pdf_file_count = len(pdf_files)

registry_pdf_count = 0
if reg_path.exists():
    df = pd.read_csv(reg_path)
    if "pdf_downloaded" in df.columns:
        registry_pdf_count = int(df["pdf_downloaded"].fillna(False).sum())

pdf_count = max(pdf_file_count, registry_pdf_count)

status_data = {
    "timestamp": time.time(),
    "pdf_file_count": pdf_file_count,
    "registry_pdf_count": registry_pdf_count,
    "pdf_count_used_for_gate": pdf_count,
    "gate_status": "READY_FOR_PARSE" if pdf_count >= MIN_PDF_REQUIRED_FOR_PARSE else "BLOCKED_NEED_MORE_PDFS"
}

with open(PROBES_DIR / "pdf_count_status.json", "w") as f:
    json.dump(status_data, f, indent=4)

print(f"Gate Status: {status_data['gate_status']} (Effective PDF count: {pdf_count})")

Gate Status: READY_FOR_PARSE (Effective PDF count: 72)


## 10. Full PDF Parse

In [12]:
if RUN_PARSE and pdf_count >= MIN_PDF_REQUIRED_FOR_PARSE:
    %cd {REPO_DIR}
    !python -m src.parse.parse_pdfs --registry corpus/document_registry.csv
    !python -m src.parse.segment_sections --parsed-dir corpus/parsed
    !python -m src.parse.extract_table_candidates --sections corpus/sections/sections.jsonl
    !python -m src.parse.validate_parse

    # Sync results back to Drive
    shutil.copy(REPO_DIR / "corpus/parse_registry.csv", REGISTRY_DIR / "parse_registry.csv")
    shutil.copy(REPO_DIR / "docs/day3_parse_report.md", REPORTS_DIR / "day3_parse_report.md")
    print("Parsing phase complete.")
elif RUN_PARSE:
    print("RUN_PARSE is True but PDF count is below threshold. Skipping.")

/content/AIGC_Fake_Detection
Parsing 72 PDFs...
Parsing complete. Successfully parsed 72 PDFs.
Registry: corpus/parse_registry.csv
Segmenting 72 files...
Segmentation complete. Saved to corpus/sections/sections.jsonl
Extracting table candidates from corpus/sections/sections.jsonl...
Extraction complete. Found 7432 candidates.
Validation complete. Report saved to docs/day3_parse_report.md
Parsing phase complete.


## 11. Final Readiness Check

In [13]:
parsed_json_count = len(list(PARSED_DIR.glob("*.json")))
sections_exists = (SECTIONS_DIR / "sections.jsonl").exists()
tables_exists = (TABLES_DIR / "table_candidates.jsonl").exists()
registry_exists = (REGISTRY_DIR / "document_registry.csv").exists()
parse_registry_exists = (REGISTRY_DIR / "parse_registry.csv").exists()

status_str = "BLOCKED_METADATA_MISSING"
if registry_exists:
    status_str = "BLOCKED_NEED_MORE_PDFS"
    if pdf_count >= 70:
        status_str = "READY_FOR_PARSE"
        if parsed_json_count >= 65 and sections_exists and tables_exists:
            status_str = "READY_FOR_NOTEBOOK_02"

final_probe = {
    "timestamp": time.time(),
    "pdf_count": pdf_count,
    "parsed_json_count": parsed_json_count,
    "sections_exists": sections_exists,
    "tables_exists": tables_exists,
    "final_status": status_str
}

with open(PROBES_DIR / "drive_data_status.json", "w") as f:
    json.dump(final_probe, f, indent=4)

print("PDF files:", pdf_file_count)
print("Registry PDFs:", registry_pdf_count)
print("Parsed JSONs:", parsed_json_count)
print("Sections exists:", sections_exists)
print("Tables exists:", tables_exists)
print(f"Final Readiness Status: {status_str}")

PDF files: 72
Registry PDFs: 72
Parsed JSONs: 72
Sections exists: True
Tables exists: True
Final Readiness Status: READY_FOR_NOTEBOOK_02


## 12. Instructions

In [14]:
if final_probe["final_status"] == "READY_FOR_NOTEBOOK_02":
    print("SUCCESS: Cloud storage is fully prepared and populated.")
    print("PROCEED: Open Notebook 02 for small-batch extraction probes.")
elif final_probe["final_status"] == "READY_FOR_PARSE":
    print("ACTION REQUIRED: PDFs are ready but parsing is incomplete. Enable RUN_PARSE and rerun.")
elif final_probe["final_status"] == "BLOCKED_NEED_MORE_PDFS":
    print("ACTION REQUIRED: Coverage is below 70 PDFs. Check fallback logs and manually upload missing papers if possible.")
else:
    print("ACTION REQUIRED: Critical metadata is missing. Ensure repo sync is successful.")

SUCCESS: Cloud storage is fully prepared and populated.
PROCEED: Open Notebook 02 for small-batch extraction probes.
